# LLM Laika
Antes de rodar: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU (T4)**.

Tudo (livros, artigos, índice de busca, pré-treino, fine-tuning e memória) fica salvo no seu
Google Drive, na pasta `LLM_Laika`. Se a sessão cair, nada se perde.

| Quero... | Rode as células |
|---|---|
| Fazer tudo do zero | 1, 2, 3, 4, 5 |
| **Só refazer o fine-tuning** (usando o pré-treino que já está no Drive) | 1, 4, 5 |
| Atualizar livros/artigos e o índice de busca | 1, 2 |
| Só conversar | 1, 5 |

In [ ]:
# 1) PREPARAR: GPU, Google Drive, código e atalhos para as pastas do Drive
import os, torch
print("GPU disponível:", torch.cuda.is_available())

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/LLM_Laika"

os.chdir("/content")
if not os.path.exists("LLM_Laika"):
    !git clone https://github.com/BrunoDevPro/LLM_Laika.git
%cd /content/LLM_Laika
!git pull -q
!pip install -q tokenizers datasets

# As pastas grandes ficam no Drive; aqui são só atalhos
for local, remote in [("data/books", "books"), ("data/wiki", "wiki"), ("data/qa_auto", "qa_auto"),
                      ("data/rag", "rag"), ("checkpoints", "checkpoints")]:
    remote = f"{DRIVE}/{remote}"
    os.makedirs(remote, exist_ok=True)
    if not os.path.islink(local):
        if os.path.isdir(local) and os.listdir(local):
            !cp -rn {local}/. {remote}/
        !rm -rf {local}
        os.symlink(remote, local)

print("No Drive:", {p: len(os.listdir(f"{DRIVE}/{p}")) for p in ["books", "wiki", "rag", "checkpoints"]})

In [ ]:
# 2) DADOS: 500 livros do Gutenberg + 100 mil artigos da Wikipédia + índice de busca (RAG)
# Só baixa o que ainda não está no Drive. A primeira vez demora (dezenas de minutos).
!python prepare_data.py --max-books 500 --wiki 100000
!python rag.py build

In [ ]:
# 3) PRÉ-TREINO (ensina português). Escolha o modo:
#   "pular"     -> usa o pré-treino que já está no Drive (padrão; se não existir, treina do zero)
#   "continuar" -> continua o pré-treino do Drive com os dados atuais (mesmo tokenizer)
#   "do_zero"   -> apaga e treina tudo de novo (necessário para o tokenizer aprender palavras novas)
MODO = "pular"

import os
existe = os.path.exists("checkpoints/pretrain_best.pt")
if MODO == "pular" and existe:
    print("Usando o pré-treino que já está no Drive.")
elif MODO == "continuar" and existe:
    !python train.py --preset colab --resume
else:
    !python train.py --preset colab

In [ ]:
# 4) FINE-TUNING (ensina a responder). Usa o pré-treino do Drive e grava checkpoints/finetuned.pt.
# Se o índice do RAG existir (célula 2), parte das perguntas é treinada com Contexto.
!python finetune.py

In [ ]:
# 5) CONVERSAR (digite 'sair' para encerrar)
# Comandos: /fontes (mostra de onde veio o contexto), /rag (liga/desliga a busca), /memoria
# A Laika lembra o que você contar (ex.: "meu nome é Ana", "lembre que amanhã tenho prova").
# Para testar só o pré-treino: sys.argv = ["chat.py", "--base"]
import sys
sys.argv = ["chat.py"]
%run chat.py